# RankResume Question Selection and Adaptive Generation

This notebook executes Phase 4 and Phase 5 only. It does not regenerate or modify the stable Question Bank.

## 1. Project and environment information

In [1]:
import json
import sys
import uuid
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'Question_Selection_Generation':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'Question_Selection_Generation'))

from Question_Selection_Generation.config import OLLAMA_MODEL
from Question_Selection_Generation.db_helper import connect, count_question_bank, ensure_schema, transaction
from Question_Selection_Generation.exam_workflow import create_exam, select_and_store_stage1, submit_stage1_scores, used_questions
from Question_Selection_Generation.llm_question_generator import check_ollama, generate_stage2_questions, sentence_count, store_stage2, validate_generated_questions
from Question_Selection_Generation.question_selector import classify_skills, determine_stage2_difficulty, load_project_vocabulary, validate_stage1_questions

print(f'Project root: {PROJECT_ROOT}')
print(f'Ollama model: {OLLAMA_MODEL}')

Project root: /home/vedant-2506/Desktop/Python_ML/Project
Ollama model: qwen3:8b


## 2. Database connection and Question Bank count

In [2]:
connection = connect()
ensure_schema(connection)
print('[DB] Connected')
bank_count = count_question_bank(connection)
print(f'[DB] Question Bank count: {bank_count}')
assert bank_count >= 5194
connection.close()

[DB] Connected
[DB] Question Bank count: 5194


## 3. Inspect relevant tables

In [3]:
connection = connect()
with connection.cursor() as cursor:
    cursor.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='public' AND table_name IN ('bankresume','job_description','technology_alias','technology_dictionary','exam_sessions','question_exam','generated_questions') ORDER BY table_name")
    print('[DB] Relevant tables:', [row[0] for row in cursor.fetchall()])
connection.close()

[DB] Relevant tables: ['bankresume', 'exam_sessions', 'generated_questions', 'job_description', 'question_exam', 'technology_alias', 'technology_dictionary']


## 4. Load and inspect an existing JD

In [4]:
JD_ID = 5
connection = connect()
with connection.cursor() as cursor:
    cursor.execute('SELECT jd_id, job_title, structured_data FROM job_description WHERE jd_id=%s', (JD_ID,))
    jd_row = cursor.fetchone()
connection.close()
assert jd_row is not None
JD = {'jd_id': jd_row[0], 'job_title': jd_row[1], **(jd_row[2] or {})}
print(json.dumps(JD, indent=2, ensure_ascii=False))

{
  "jd_id": 5,
  "job_title": "senior java developer",
  "expected_skills": [
    "java",
    "spring boot",
    "sql",
    "git",
    "c++",
    "mysql",
    "ci/cd",
    "communication",
    "problem solving",
    "teamwork"
  ],
  "expected_education": [
    {
      "degree": "b.sc computer science or b.e computer engineering",
      "percentage/mark": "60%"
    }
  ],
  "expected_experience": [
    {
      "skill": [],
      "years": "5 years"
    }
  ],
  "expected_certificate": []
}


## 5. Extract Core CS and Changing Technology skills

In [5]:
connection = connect()
core_skills, changing_skills = classify_skills(JD.get('expected_skills', []), load_project_vocabulary(connection))
connection.close()
print('[SKILL] Core skills:', core_skills)
print('[SKILL] Changing Tech skills:', changing_skills)
assert core_skills
assert changing_skills

[SKILL] Core skills: ['SQL', 'Git / Version Control']
[SKILL] Changing Tech skills: ['Spring Boot', 'CI/CD']


## 6. Select and validate six Question Bank questions

In [6]:
with transaction() as connection:
    exam_id, exam_info = create_exam(connection, JD_ID)
    stage1_questions, selected_core, selected_changing = select_and_store_stage1(connection, exam_id, JD_ID)

stage1_errors = validate_stage1_questions(stage1_questions)
assert not stage1_errors, stage1_errors
print(f'[SELECT] Exam: {exam_id}')
print('[SELECT] Easy:', sum(q.difficulty == 'Easy' for q in stage1_questions))
print('[SELECT] Medium:', sum(q.difficulty == 'Medium' for q in stage1_questions))
print('[SELECT] Hard:', sum(q.difficulty == 'Hard' for q in stage1_questions))

[SELECT] Exam: d299d5bd-5b44-4551-a382-bd59d272769f
[SELECT] Easy: 3
[SELECT] Medium: 2
[SELECT] Hard: 1


## 7. Display selected questions

In [7]:
for order, item in enumerate(stage1_questions, 1):
    print(f'Q{order} [{item.difficulty}] {item.subject} / {item.topic}: {item.question}')

Q1 [Hard] Git / Version Control / branches: In Git / Version Control, What trade-offs should guide the choice of branches for a batch job?
Q2 [Easy] SQL / subqueries: In SQL, How does subqueries work internally when it is used in a multi-user application?
Q3 [Easy] SQL / SELECT and filtering: In SQL, What is SELECT and filtering, and what problem does it address in a concurrent server?
Q4 [Easy] SQL / GROUP BY and HAVING: In SQL, What is GROUP BY and HAVING, and what problem does it address in a library used by several teams?
Q5 [Medium] Git / Version Control / branches: In Git / Version Control, What common failure does branches prevent or expose in a system under test, and why?
Q6 [Medium] SQL / GROUP BY and HAVING: In SQL, What common failure does GROUP BY and HAVING prevent or expose in a command-line utility, and why?


## 8. Validate adaptive difficulty boundaries

In [8]:
boundary_cases = {100: {'Hard': 4}, 81: {'Hard': 4}, 80: {'Hard': 2, 'Easy': 2}, 60: {'Hard': 2, 'Easy': 2}, 59: {'Easy': 4}}
for score, expected in boundary_cases.items():
    actual = determine_stage2_difficulty(score)
    print(score, actual)
    assert actual == expected
stage1_result = submit_stage1_scores([82, 77, 91, 68, 74, 86], {'SQL': 68, 'Git': 82})
print('[ADAPTIVE]', stage1_result)

100 {'Hard': 4}
81 {'Hard': 4}
80 {'Hard': 2, 'Easy': 2}
60 {'Hard': 2, 'Easy': 2}
59 {'Easy': 4}
[ADAPTIVE] {'average': 79.66666666666667, 'difficulty_distribution': {'Hard': 2, 'Easy': 2}, 'skill_performance': {'SQL': 68, 'Git': 82}}


## 9. Check Ollama and qwen3:8b

In [9]:
ollama_available = check_ollama(OLLAMA_MODEL)
print(f'[OLLAMA] Model: {OLLAMA_MODEL}')
print('[OLLAMA] Connection:', 'SUCCESS' if ollama_available else 'FAILED')
assert ollama_available, f'{OLLAMA_MODEL} is not available'

[OLLAMA] Model: qwen3:8b
[OLLAMA] Connection: SUCCESS


## 10. Generate four Changing-Tech questions

In [10]:
average_score = stage1_result['average']
with connect() as connection:
    used = used_questions(connection, exam_id)
generated_questions = generate_stage2_questions(JD, changing_skills, average_score, used, stage1_result['skill_performance'])
print(f'[LLM] Generated questions: {len(generated_questions)}')
for item in generated_questions:
    print(f'[{item.difficulty}] {item.skill}: {item.question}')
    print(f'Sentences: {sentence_count(item.reference_answer)}; concepts: {item.expected_concepts}')
assert len(generated_questions) == 4

KeyboardInterrupt: 

## 11. Validate generated JSON, scope, answers, and duplicates

In [ ]:
distribution = stage1_result['difficulty_distribution']
assert all(item.skill in changing_skills for item in generated_questions)
assert all(5 <= sentence_count(item.reference_answer) <= 6 for item in generated_questions)
assert len({item.question.lower().strip() for item in generated_questions}) == 4
actual_distribution = {difficulty: sum(item.difficulty == difficulty for item in generated_questions) for difficulty in ('Easy', 'Medium', 'Hard')}
expected_distribution = {difficulty: distribution.get(difficulty, 0) for difficulty in ('Easy', 'Medium', 'Hard')}
assert actual_distribution == expected_distribution, (actual_distribution, expected_distribution)
print('[VALIDATION] Structured output, Changing-Tech scope, answer length, and same-batch uniqueness passed')

## 12. Store generated questions and exam mapping

In [ ]:
with transaction() as connection:
    generated_ids = store_stage2(connection, exam_id, JD_ID, generated_questions, OLLAMA_MODEL)
print('[DB] Generated question IDs:', generated_ids)
assert len(generated_ids) == 4

## 13. Test idempotency and database records

In [ ]:
with transaction() as connection:
    second_insert = store_stage2(connection, exam_id, JD_ID, generated_questions, OLLAMA_MODEL)
    with connection.cursor() as cursor:
        cursor.execute('SELECT question_order, stage, source, difficulty FROM question_exam WHERE exam_id=%s ORDER BY question_order', (str(exam_id),))
        mapping = cursor.fetchall()
        cursor.execute('SELECT status, duration_minutes FROM exam_sessions WHERE exam_id=%s', (str(exam_id),))
        session = cursor.fetchone()
print('[DB] Second Stage-2 insertion:', second_insert)
print('[DB] Mapping:', mapping)
print('[DB] Session:', session)
assert second_insert == []
assert len(mapping) == 10
assert [row[2] for row in mapping[:6]] == ['Question Bank'] * 6
assert [row[2] for row in mapping[6:]] == ['LLM'] * 4
assert session[1] == 30

## 14. Final validation report

In [ ]:
with connect() as connection:
    final_bank_count = count_question_bank(connection)
report = {
    'database': 'Connected',
    'question_bank_count': final_bank_count,
    'stage1_questions': 6,
    'stage1_distribution': {'Easy': 3, 'Medium': 2, 'Hard': 1},
    'ollama_model': OLLAMA_MODEL,
    'ollama': 'Connected',
    'changing_tech_skills': changing_skills,
    'generated_questions': 4,
    'generated_question_validation': 'PASS',
    'duplicate_validation': 'PASS',
    'database_storage': 'PASS',
    'single_timer_minutes': 30,
    'end_to_end_test': 'PASS',
    'notebook_execution': 'PASS',
    'warnings': [],
}
print(json.dumps(report, indent=2, ensure_ascii=False))
assert final_bank_count >= 5194